# Unit 4 · AI-Assisted Visualization — Case-Based Activity
**Case:** *Auditing AutoViz at SwiftBite*

SwiftBite is a food-delivery start-up. Its analytics lead has bought **AutoViz**, an automated visualization tool, and wants to
roll it out to every analyst. You have been asked to **audit** it first: check how it reads the data, how it picks encodings, how its
recommendations are ranked, and what happens when it generates charts from plain-language questions.

**How to work**
* Run the cells from top to bottom. Do **not** edit the setup cell.
* Wherever you see `# TODO`, replace the placeholder (`None` or `""`) with your answer or code.
* Written answers: keep them to two or three sentences and use ideas from the lecture.

In [ ]:
import math, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 11, "axes.titleweight": "bold"})
pd.set_option("display.max_columns", 20, "display.width", 140)

# ---------------- synthetic SwiftBite orders (do not edit) ----------------
rng = np.random.default_rng(7)
n = 900
cities = ["Bengaluru", "Mumbai", "Delhi", "Chennai"]
cuisines = ["North Indian", "South Indian", "Chinese", "Italian", "Desserts"]
zones = {"Bengaluru": [560001, 560034, 560066], "Mumbai": [400001, 400050, 400070],
         "Delhi": [110001, 110016, 110045], "Chennai": [600001, 600020, 600042]}
city_effect = {"Bengaluru": 2, "Mumbai": 4, "Delhi": 1, "Chennai": -4}

dates = pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n), unit="D")
city = rng.choice(cities, n, p=[0.35, 0.25, 0.25, 0.15])
distance = np.clip(rng.gamma(2.5, 1.4, n), 0.5, 14).round(1)
prep = np.clip(rng.normal(14, 3.5, n), 5, None)
delivery = (prep + distance * 3.0 + np.array([city_effect[c] for c in city]) + rng.normal(0, 3, n)).round(1)
sat_score = 4.2 - 0.05 * delivery + rng.normal(0, 0.55, n)
satisfaction = pd.cut(sat_score, [-np.inf, 2.0, 2.8, 3.5, np.inf], labels=["Poor", "Fair", "Good", "Excellent"]).astype(str)

df = pd.DataFrame({
    "order_id": np.arange(100001, 100001 + n),
    "order_date": dates,
    "month": dates.strftime("%b"),
    "city": city,
    "cuisine": rng.choice(cuisines, n),
    "zone_code": [int(rng.choice(zones[c])) for c in city],
    "distance_km": distance,
    "delivery_time_min": delivery,
    "order_value": (rng.gamma(3.0, 110, n) + 40).round(0),
    "satisfaction": satisfaction,
}).sort_values("order_date").reset_index(drop=True)

MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
df.head()

---
## Case 1 · What does AutoViz *see*? (Step 1 of the pipeline: data abstraction)
AutoViz inspects the columns and guesses an attribute type for each one. Its guesses are below.

In [ ]:
tool_types = {
    "order_id":          "Quantitative discrete",
    "order_date":        "Date / time",
    "month":             "Categorical unordered",
    "city":              "Categorical unordered",
    "cuisine":           "Categorical unordered",
    "zone_code":         "Quantitative discrete",
    "distance_km":       "Quantitative continuous",
    "delivery_time_min": "Quantitative continuous",
    "order_value":       "Quantitative continuous",
    "satisfaction":      "Categorical unordered",
}
OPTIONS = ["Quantitative continuous", "Quantitative discrete", "Categorical unordered", "Categorical ordered", "Date / time"]
pd.DataFrame({"column": list(tool_types), "AutoViz's guess": list(tool_types.values()), "sample": [str(df[c].iloc[0]) for c in tool_types]})

**1a.** Four of AutoViz's guesses are wrong in a way that would damage the chart it builds. Provide the correct type for each, choosing from `OPTIONS`.

In [ ]:
corrections = {
    "order_id":     None,   # TODO
    "month":        None,   # TODO
    "zone_code":    None,   # TODO
    "satisfaction": None,   # TODO
}
assert all(v in OPTIONS for v in corrections.values()), "Choose each answer from OPTIONS"

**1b.** Because AutoViz treats `month` as unordered text, its "orders per month" chart comes out alphabetical (Apr, Aug, Dec …).
Build `orders_by_month`: the number of orders in each month, **in calendar order**.

In [ ]:
tool_version = df.groupby("month").size()      # what AutoViz drew (alphabetical)

orders_by_month = None   # TODO: same counts, but indexed Jan -> Dec

assert list(orders_by_month.index) == MONTHS, "Months must run Jan -> Dec"
assert int(orders_by_month.sum()) == len(df)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3), sharey=True)
ax[0].bar(tool_version.index, tool_version.values, color="#C44E52"); ax[0].set_title("AutoViz (alphabetical)")
ax[1].bar(orders_by_month.index, orders_by_month.values, color="#4C72B0"); ax[1].set_title("Your fix (calendar order)")
plt.tight_layout(); plt.show()

---
## Case 2 · Is AutoViz's encoding choice any good? (Marks, channels, expressiveness, effectiveness)
For the next dashboard, AutoViz proposed five encodings. The finance team says **`order_value` is the most important field** on the dashboard.

In [ ]:
proposals = pd.DataFrame([
    ("P1", "delivery_time_min", "Quantitative",        "bar length"),
    ("P2", "city",              "Categorical (unordered)", "colour luminance ramp (light -> dark)"),
    ("P3", "order_value",       "Quantitative (most important)", "bubble area"),
    ("P4", "satisfaction",      "Categorical (ordered Poor < ... < Excellent)", "marker shape"),
    ("P5", "distance_km",       "Quantitative",        "y position"),
], columns=["id", "field", "type", "AutoViz's channel"]).set_index("id")
proposals

**2a.** Give a verdict for every proposal using exactly one of:
`"Accept"` · `"Reject (not expressive)"` · `"Accept but rank low"`

In [ ]:
verdicts = {
    "P1": None,   # TODO
    "P2": None,   # TODO
    "P3": None,   # TODO
    "P4": None,   # TODO
    "P5": None,   # TODO
}
VERDICT_OPTIONS = {"Accept", "Reject (not expressive)", "Accept but rank low"}
assert all(v in VERDICT_OPTIONS for v in verdicts.values()), "Use one of the three verdict strings"

**2b.** AutoViz encodes average `order_value` as **bubble area** (Stevens' exponent for area: *n = 0.7*).
Zone A's average order is ₹400 and Zone B's is ₹1000. Compute, using `S = Iⁿ`:
1. how many times bigger B **looks** than A when drawn as area,
2. how many times larger B's *area* would have to be for B to **look** 2.5× bigger,
3. how many times bigger B looks if the same data were drawn as **bar length** (*n = 1.0*).

In [ ]:
n_area, n_length = 0.7, 1.0
true_ratio = 1000 / 400

looks_ratio_area   = None   # TODO (1)
area_ratio_needed  = None   # TODO (2)
looks_ratio_length = None   # TODO (3)

print(f"True ratio: {true_ratio}")
print("Looks (area):", looks_ratio_area, "| area needed:", area_ratio_needed, "| Looks (length):", looks_ratio_length)

**2c.** Now let AutoViz's greedy (APT-style) encoder pick for the whole dashboard. Cost = Σ (field weight × channel rank); lower is better.
Candidates that are **not expressive are rejected before scoring.** Write `encoding_cost(enc)` so it returns `None` for a rejected candidate, otherwise the cost.
Then name the best candidate.

In [ ]:
CHANNEL_RANK = {"x position": 1, "y position": 1, "length": 3, "angle": 4, "area": 5, "luminance": 7,   # magnitude (ordered data)
                "region": 1, "hue": 2, "shape": 4}                                                       # identity (categorical data)
weights = {"delivery_time_min": 5, "distance_km": 3, "order_value": 2, "city": 1}
kinds   = {"delivery_time_min": "Q", "distance_km": "Q", "order_value": "Q", "city": "N"}

def is_expressive(kind, channel):
    if kind == "N": return channel not in {"luminance", "length", "area", "angle"}   # must not look ordered
    if kind == "Q": return channel not in {"hue", "shape", "region"}                 # must look ordered
    raise ValueError(kind)

candidates = {
    "E1": {"delivery_time_min": "y position", "distance_km": "x position", "order_value": "area",      "city": "hue"},
    "E2": {"delivery_time_min": "y position", "distance_km": "x position", "order_value": "luminance", "city": "shape"},
    "E3": {"delivery_time_min": "area",       "distance_km": "x position", "order_value": "y position","city": "hue"},
    "E4": {"order_value": "y position", "delivery_time_min": "x position", "distance_km": "area",      "city": "hue"},
    "E5": {"delivery_time_min": "y position", "distance_km": "x position", "order_value": "area",      "city": "luminance"},
}
pd.DataFrame(candidates).T

In [ ]:
def encoding_cost(enc):
    # TODO: return None if ANY field -> channel pair is not expressive (use is_expressive and kinds)
    #       otherwise return the sum of weights[field] * CHANNEL_RANK[channel]
    pass

costs = {name: encoding_cost(enc) for name, enc in candidates.items()}
best_encoding = None   # TODO: name of the valid candidate with the lowest cost

print(costs)
assert costs["E5"] is None, "E5 should be rejected before scoring"
assert all(isinstance(costs[k], (int, float)) for k in ["E1", "E2", "E3", "E4"])

---
## Case 3 · Which recommender should SwiftBite trust? (Rule/constraint-based vs ML-based, evaluation)
### 3a · A Draco-style constraint model
The task is *"show how `distance_km` relates to `delivery_time_min` for the 4 cities."* The soft-constraint weights are below,
and each candidate chart lists how many times it violates each soft constraint. A candidate that breaks a **hard** constraint is `None`.

In [ ]:
soft_w = {"s1": 6,   # aggregation hides points (task = correlation)
          "s2": 2,   # categorical field on colour
          "s3": 3,   # quantitative field on size
          "s4": 2}   # each encoding beyond 2 (clutter)

violations = {
    "A: scatter  x=distance, y=delivery_time, colour=city":                 {"s2": 1, "s4": 1},
    "B: A + size=order_value":                                              {"s2": 1, "s3": 1, "s4": 2},
    "C: bar of mean delivery_time per city":                                {"s1": 1},
    "D: scatter  x=distance, y=delivery_time, shape=order_value":           None,   # quantitative on shape: hard violation
}

In [ ]:
def draco_cost(v, w):
    # TODO: None for an invalid (hard-violating) candidate, else sum of count * weight
    pass

default_costs = {k: draco_cost(v, soft_w) for k, v in violations.items()}
print(default_costs)

# The team now decides colour is "too busy": raise w(s2) from 2 to 5
new_w = {**soft_w, "s2": 5}
new_costs = {k: draco_cost(v, new_w) for k, v in violations.items()}
print(new_costs)

winner_default = None   # TODO: first letter ("A", "B", "C") of the lowest-cost valid candidate under soft_w
winner_new     = None   # TODO: same, under new_w

### 3b · Comparing two recommenders
AutoViz (**R1**) and a competitor, QuickCharts (**R2**), were each tested on 8 datasets. The list shows the **rank at which the expert's chart appeared**
(`None` = not in the list). The UI always shows a **gallery of 3 charts**.

In [ ]:
ranks = {"R1 (AutoViz)":    [1, 1, 1, None, 1, None, 1, 2],
         "R2 (QuickCharts)": [2, 1, 2, 3, 2, 2, 3, 1]}

In [ ]:
def hit_at_k(r, k):
    # TODO: fraction of datasets where the expert chart is ranked within the top k
    pass

def mrr(r):
    # TODO: mean reciprocal rank (use 0 for None)
    pass

assert abs(hit_at_k([1, 2, None, 4], 2) - 0.5) < 1e-9
assert abs(mrr([1, None]) - 0.5) < 1e-9

report = pd.DataFrame({name: {"Hit@1": hit_at_k(r, 1), "Hit@3": hit_at_k(r, 3), "MRR": mrr(r)} for name, r in ranks.items()}).round(3)
display(report)

better_for_gallery = None   # TODO: "R1" or "R2"

### 3c · Choosing the approach
SwiftBite has **no archive of past charts** and its compliance team insists that for every recommendation the tool must be able to **say exactly why** it chose that chart.
Which approach fits best? Set `answer_3c` to `"A"`, `"B"` or `"C"`.

* **A.** An ML model such as VizML, trained on a large corpus of charts
* **B.** A rule/constraint-based system such as Show Me or Draco
* **C.** Either; explainability does not differ between them

In [ ]:
answer_3c = None   # TODO

---
## Case 4 · Same data, different task (Task- and data-aware recommendation)
### 4a · Task-aware scoring
Data: **monthly average delivery time for the 4 cities.** Fit scores for each candidate chart are below.
`score = 0.7 × task_fit + 0.3 × data_fit`.

In [ ]:
fits = pd.DataFrame({"data_fit":         [0.90, 0.80, 0.70, 0.30],
                     "task_fit_trend":   [0.95, 0.55, 0.60, 0.05],
                     "task_fit_compare": [0.40, 0.90, 0.65, 0.60]},
                    index=["Line", "Grouped bar", "Heatmap", "Pie"])
W_TASK, W_DATA = 0.7, 0.3

In [ ]:
# TODO: add two columns, score_trend and score_compare, using W_TASK and W_DATA
fits["score_trend"] = None
fits["score_compare"] = None

best_trend   = None   # TODO: name of the best chart when the task is "see the trend"
best_compare = None   # TODO: name of the best chart when the task is "compare cities overall"
display(fits)

### 4b · Fix the mini recommender
This is the rule-based recommender from the lecture hands-on. Run the test table first and see what goes wrong, then fix it:

1. `zone_code` has 12 distinct numbers, so `col_type` calls it `"Q"` and the tool will happily draw a scatter plot of pin-codes. Make columns whose **name ends in `_id` or `_code`** count as `"N"`.
2. Several useful requests return `"ask the user"`. Add rules so that **a categorical + a quantitative field with task `distribution`** and **one quantitative field with task `outliers`** each return a sensible chart.

In [ ]:
def col_type(s, name=""):
    if pd.api.types.is_datetime64_any_dtype(s): return "T"
    # TODO 1: identifiers / codes are labels, not numbers -> "N"
    if pd.api.types.is_numeric_dtype(s) and s.nunique() > 10: return "Q"
    return "N"

RULES = {("NQ", "compare"):       "bar",
         ("QT", "trend"):         "line",
         ("QQ", "correlation"):   "scatter",
         ("Q",  "distribution"):  "histogram",
         ("NQ", "part-to-whole"): "pie"}
# TODO 2: add the two missing rules to RULES

def recommend(df, fields, task="compare"):
    t = "".join(sorted(col_type(df[f], f) for f in fields))
    return RULES.get((t, task), "ask the user")

tests = [(["city", "order_value"], "compare"), (["order_date", "order_value"], "trend"),
         (["distance_km", "delivery_time_min"], "correlation"), (["delivery_time_min"], "distribution"),
         (["zone_code", "order_value"], "correlation"),
         (["city", "delivery_time_min"], "distribution"), (["delivery_time_min"], "outliers")]
display(pd.DataFrame([(f, t, recommend(df, f, t)) for f, t in tests], columns=["fields", "task", "recommended"]))

assert recommend(df, ["zone_code", "order_value"], "correlation") == "ask the user", "pin-codes must not be treated as numbers"
assert recommend(df, ["city", "delivery_time_min"], "distribution") != "ask the user", "add the distribution rule"
assert recommend(df, ["delivery_time_min"], "outliers") != "ask the user", "add the outliers rule"
assert recommend(df, ["city", "order_value"], "compare") == "bar"

---
## Case 5 · Why a gallery? (Visualization search and recommendation)
An analyst selects **4 fields** (3 quantitative, 1 categorical). AutoViz can use **6 channels** (x, y, colour, size, shape, facet),
**4 mark types** (point, bar, line, area), and each quantitative field can be **raw, binned, mean or sum** (4 options).

**5a.** How many candidate specifications exist? Then suppose the expressiveness (hard) constraints remove **90 %** of them — how many are left to rank?

In [ ]:
total_specs   = None   # TODO
after_pruning = None   # TODO

print(f"{total_specs:,} specs  ->  {after_pruning:,} after pruning")

**5b.** The product manager asks: *"Why does the tool show a gallery of several top charts per field set, in the style of Voyager, instead of just the single best chart?"*
Answer in two or three sentences, using ideas from the lecture.

In [ ]:
answer_5b = ""   # TODO

---
## Case 6 · "Just ask in plain English" (Text-to-chart and the limits of automation)
AutoViz's newest feature is text-to-chart. The prompt it builds contains the **question, the data schema and a few sample rows**; the LLM answers with code, which AutoViz runs.
An analyst typed: **"Which city has the happiest customers?"** and got three different responses over three attempts. Run the cell below to see each one.

In [ ]:
snippets = {
"A": """
fig, ax = plt.subplots()
t = df.groupby('city')['happiness_score'].mean().sort_values()
ax.barh(t.index, t.values)
ax.set_title('Happiest cities')
""",
"B": """
fig, ax = plt.subplots()
df2 = df.assign(score=df['satisfaction'].astype('category').cat.codes)
t = df2.groupby('city')['score'].mean().sort_values()
ax.barh(t.index, t.values)
ax.set_xlabel('Satisfaction score')
ax.set_title('Mean satisfaction by city')
""",
"C": """
fig, ax = plt.subplots()
m = {'Poor': 1, 'Fair': 2, 'Good': 3, 'Excellent': 4}
t = df['satisfaction'].map(m).groupby(df['city']).sum().sort_values()
ax.barh(t.index, t.values)
ax.set_xlabel('Satisfaction score')
ax.set_title('Average satisfaction by city')
""",
}

def run_snippet(name):
    print(f"----- Response {name} -----")
    try:
        exec(snippets[name], {"df": df, "pd": pd, "np": np, "plt": plt})
        plt.show()
    except Exception as e:
        print("The generated code crashed:", type(e).__name__, e)

for k in snippets:
    run_snippet(k)

**6a.** Diagnose each response. Choose exactly one label from `FAILURE_MODES` for each of A, B, C.
*Hint: print `df["satisfaction"].astype("category").cat.categories` and compare the group sizes with `df.city.value_counts()`.*

In [ ]:
FAILURE_MODES = ["hallucinated column", "wrong aggregation", "ordinal order ignored", "misleading chart type", "no problem"]

diagnosis = {
    "A": None,   # TODO
    "B": None,   # TODO
    "C": None,   # TODO
}
assert all(v in FAILURE_MODES for v in diagnosis.values()), "Choose from FAILURE_MODES"

**6b.** Write the corrected analysis. `mean_satisfaction_by_city(df)` must map the ordered levels **Poor < Fair < Good < Excellent** to 1–4 explicitly and return the **mean** score per city
as a Series (sorted ascending).

In [ ]:
def mean_satisfaction_by_city(df):
    order = {"Poor": 1, "Fair": 2, "Good": 3, "Excellent": 4}
    # TODO: map df["satisfaction"] with `order`, take the MEAN per city, sort ascending, return the Series
    pass

result = mean_satisfaction_by_city(df)
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(result.index, result.values, color="#4C72B0")
ax.set_xlabel("Mean satisfaction (1 = Poor ... 4 = Excellent)"); ax.set_title("Mean satisfaction by city")
plt.show()

_ref = df["satisfaction"].map({"Poor": 1, "Fair": 2, "Good": 3, "Excellent": 4}).groupby(df["city"]).mean().sort_values()
assert list(result.index) == list(_ref.index) and np.allclose(result.values, _ref.values), "Check the mapping, the aggregation and the sorting"

**6c.** Apply the lecture's **human-in-the-loop checklist** to **Response C's chart**. Set each item to `True` (passes) or `False` (fails).
Then write your recommendation (two or three sentences): should SwiftBite roll AutoViz out to every analyst as it is, and where must a human stay in the loop?

In [ ]:
checklist_for_C = {
    "1 Right data? (correct fields, filters, aggregation)":   None,   # TODO
    "2 Right types? (IDs, ordered levels handled correctly)": None,   # TODO
    "3 Expressive? (no fake order, no shape for numbers)":    None,   # TODO
    "4 Effective? (key quantity on position/length)":         None,   # TODO
    "5 Fits the task? (compare cities)":                      None,   # TODO
    "6 Low cognitive cost? (readable at a glance)":           None,   # TODO
}
assert all(isinstance(v, bool) for v in checklist_for_C.values()), "Use True or False"

final_recommendation = ""   # TODO

---
### Before you finish
Run the cell below to see whether any answer has been left blank.

In [ ]:
_required = ["corrections", "orders_by_month", "verdicts", "looks_ratio_area", "area_ratio_needed", "looks_ratio_length", "best_encoding",
             "winner_default", "winner_new", "better_for_gallery", "answer_3c", "best_trend", "best_compare",
             "total_specs", "after_pruning", "answer_5b", "diagnosis", "checklist_for_C", "final_recommendation"]
_blank = []
for _name in _required:
    _v = globals().get(_name)
    if _v is None or (isinstance(_v, str) and not _v.strip()) or (isinstance(_v, dict) and any(x is None for x in _v.values())):
        _blank.append(_name)
print("All answers filled in." if not _blank else "Still blank: " + ", ".join(_blank))